# **Laboratório 5: Sincronização por Condição (Variáveis de Condição)**

**Disciplina:** Programação Concorrente (ICP-361) - UFRJ

## **Introdução**

O objetivo deste Laboratório é introduzir o mecanismo de sincronização por condição usando variáveis de condição da biblioteca Pthread.


In [ ]:
!gcc --version

gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
Copyright (C) 2021 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



## **Preparação do Ambiente**

Como o Google Colab roda em Linux, podemos compilar e executar códigos C nativamente. Vamo verificar a versão do compilador gcc.


## **Atividade 1: Barreira**

**Objetivo:** Experimentar o padrão de sincronização coletiva (barreira).

O código abaixo simula o arquivo `barreira.c` mencionado no roteiro. Ele implementa uma barreira onde as threads devem esperar até que todas cheguem ao ponto de sincronização antes de prosseguir.

### **1. Criar o código da Barreira**

Execute a célula abaixo para criar o arquivo `barreira.c`.

In [ ]:
%%writefile barreira.c
#include <pthread.h>
#include <stdio.h>
#include <stdlib.h>
#include <unistd.h>

#define NTHREADS 5
#define PASSOS 4

pthread_barrier_t barreira_global;

void *tarefa(void *arg) {
    int id = *(int *)arg;
    int b;

    for (int i = 0; i < PASSOS; i++) {
        printf("Thread %d: passo %d\n", id, i);

        for (b = 0; b < 1000000; b++);

        int ret = pthread_barrier_wait(&barreira_global);
        if (ret != 0 && ret != PTHREAD_BARRIER_SERIAL_THREAD) {
            printf("Erro na barreira da thread %d\n", id);
            pthread_exit(NULL);
         }
    }

    pthread_exit(NULL);
}

int main(int argc, char *argv[]) {
    pthread_t threads[NTHREADS];
    int id[NTHREADS];

    if (pthread_barrier_init(&barreira_global, NULL, NTHREADS-3) != 0) {
        printf("Erro ao inicializar a barreira.\n");
        return 1;
    }

    for (int i = 0; i < NTHREADS; i++) {
        id[i] = i;
        pthread_create(&threads[i], NULL, tarefa, (void *)&id[i]);
    }

    for (int i = 0; i < NTHREADS; i++) {
        pthread_join(threads[i], NULL);
    }

    pthread_barrier_destroy(&barreira_global);
    printf("FIM.\n");
    return 0;
}

Overwriting barreira.c


### **2. Execução (Sem Barreira)**

Compile e execute o programa. Verifique se as threads estão sincronizadas ou desordenadas.

In [ ]:
!gcc -o barreira barreira.c -lpthread
!./barreira

Thread 0: passo 0
Thread 2: passo 0
Thread 1: passo 0
Thread 3: passo 0
Thread 2: passo 1
Thread 0: passo 1
Thread 1: passo 1
Thread 4: passo 0
Thread 0: passo 2
Thread 3: passo 1
Thread 1: passo 2
Thread 4: passo 1
Thread 2: passo 2
Thread 4: passo 2
Thread 1: passo 3
Thread 3: passo 2
Thread 3: passo 3
Thread 2: passo 3
Thread 4: passo 3
Thread 0: passo 3
FIM.


### **3. Execução (Com Barreira)**

**Instrução:** Edite a célula do código `barreira.c` acima, **descomentando** as linhas comentadas. Depois, rode a célula de compilação abaixo novamente. Qual foi a diferença?

In [ ]:
# Recompile e execute após alterar o código
!gcc -o barreira barreira.c -lpthread
!./barreira

Thread 0: passo 0
Thread 1: passo 0
Thread 1: passo 1
Thread 2: passo 0
Thread 3: passo 0
Thread 3: passo 1
Thread 4: passo 0
Thread 0: passo 1
Thread 2: passo 1
Thread 4: passo 1
Thread 1: passo 2
Thread 1: passo 3
Thread 0: passo 2
Thread 3: passo 2
Thread 3: passo 3
Thread 4: passo 2
Thread 0: passo 3
Thread 2: passo 2
Thread 2: passo 3
Thread 4: passo 3
FIM.


### **4. Execução (Com Barreira e número diferente de threads)**

**Instrução:** Edite a célula do código `barreira.c` acima, alterando a linha para:

```c
if (pthread_barrier_init(&barreira_global, NULL, NTHREADS+1) != 0)
```
e depois para:

```c
if (pthread_barrier_init(&barreira_global, NULL, NTHREADS-3) != 0)
```

  - Verifique o que acontece em cada caso.

In [ ]:
# Recompile e execute após alterar o código
!gcc -o barreira barreira.c -lpthread
!./barreira

Thread 0: passo 0
Thread 2: passo 0
Thread 2: passo 1
Thread 1: passo 0
Thread 3: passo 0
Thread 4: passo 0
Thread 1: passo 1
Thread 4: passo 1
Thread 0: passo 1
Thread 3: passo 1
Thread 4: passo 2
Thread 3: passo 2
Thread 1: passo 2
Thread 2: passo 2
Thread 4: passo 3
Thread 3: passo 3
Thread 0: passo 2
Thread 2: passo 3
Thread 0: passo 3
Thread 1: passo 3
FIM.


In [ ]:
# Recompile e execute após alterar o código
!gcc -o barreira barreira.c -lpthread
!./barreira

Thread 0: passo 0
Thread 1: passo 0
Thread 1: passo 1
Thread 2: passo 0
Thread 0: passo 1
Thread 3: passo 0
Thread 0: passo 2
Thread 4: passo 0
Thread 0: passo 3
Thread 2: passo 1
Thread 4: passo 1
Thread 3: passo 1
Thread 1: passo 2
Thread 4: passo 2
Thread 2: passo 2
Thread 3: passo 2
Thread 2: passo 3
Thread 4: passo 3
Thread 1: passo 3
Thread 3: passo 3
FIM.


## **Atividade 2: Hello Bye (Ordem de Execução)**

**Objetivo:** Usar variáveis de condição para controlar a ordem: imprimir "HELLO" antes de "BYEBYE".

### **1. Criar o código `hellobye.c`**

In [ ]:
%%writefile hellobye.c
/* Disciplina: Programação Concorrente */
/* Profa.: Silvana Rossetto */
/* Laboratório: 5 */
/* Codigo: Uso de variáveis de condição e suas operações básicas para sincronização por condição */
/* Condição lógica da aplicação: uma thread A deve imprimir HELLO antes da thread B imprimir BYEBYE */

#include <pthread.h>
#include <stdio.h>
#include <stdlib.h>

#define NTHREADS  2

/* Variaveis globais */
short int hello = 0;
pthread_mutex_t mutex;
pthread_cond_t cond;

/* Thread A */
void *A (void *t) {
  printf("A: Comecei\n");

  //faz uma computação qualquer...

  //executa ''hello'' (OU vai para o estado 'hello')
  printf("HELLO\n");

  //regitra transicao de estado (para o caso da outra thread ainda não ter chegado ao ponto que deve verificar o estado)
  pthread_mutex_lock(&mutex);
  hello=1; //registra a transição de estado
  //sinaliza outra thread (para o caso da outra thread já estar bloqueada)
  printf("A: vai sinalizar a condicao \n");
  pthread_cond_signal(&cond);
  pthread_mutex_unlock(&mutex);

  //faz outra computação qualquer...

  pthread_exit(NULL);
}

/* Thread B */
void *B (void *t) {
  printf("B: Comecei\n");

  //faz uma computação qualquer...

  //verifica se pode executar ''byebye''
  pthread_mutex_lock(&mutex);
  if (!hello) { // estado não transicionou ainda...
     printf("B: vai se bloquear para aguardar transicao de estado\n");
     pthread_cond_wait(&cond, &mutex); //bloqueia até estado valido
     printf("B: sinal recebido e mutex realocado \n");
  }
  pthread_mutex_unlock(&mutex);

  //executa ''byebye''
  printf("BYEBYE\n");

  //faz outra computação qualquer...

  pthread_exit(NULL);
}


/* Funcao principal */
int main(int argc, char *argv[]) {
  pthread_t threads[NTHREADS];

  /* Inicializa o mutex (lock de exclusao mutua) e a variavel de condicao */
  pthread_mutex_init(&mutex, NULL);
  pthread_cond_init (&cond, NULL);

  /* Cria as threads */
    pthread_create(&threads[0], NULL, A, NULL);
    pthread_create(&threads[1], NULL, B, NULL);

  /* Espera todas as threads completarem */
  for (int i = 0; i < NTHREADS; i++) {
    pthread_join(threads[i], NULL);
  }
  printf ("\nFIM\n");

  /* Desaloca variaveis e termina */
  pthread_mutex_destroy(&mutex);
  pthread_cond_destroy(&cond);
  return 0;
}

Overwriting hellobye.c



### **2. Execução e Testes**

1. Abra o arquivo hello bye.c e identifique qual é o requisito lógico/condicional da aplicação  (qual é a ordem de impressão requerida para as expressões HELLO e BYE). Acompanhe a explanação.
2. Execute a aplicação várias vezes e verifique se o requisito lógico é sempre cumprido.
3. Inverta a ordem de criação das threads A e B e execute o programa novamente. Verifique os resultados.

In [ ]:
!gcc -o hellobye hellobye.c -lpthread
!./hellobye

A: Comecei
HELLO
A: vai sinalizar a condicao 
B: Comecei
BYEBYE

FIM


## **Atividade 3: Hello 2 Bye**

**Objetivo:** Duas threads imprimem "HELLO", uma thread espera ambas para imprimir "BYEBYE".

### **1. Criar código `hello2bye.c`**

In [ ]:
%%writefile hello2bye.c
#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>

/* Variaveis globais */
int x = 0;
pthread_mutex_t x_mutex;
pthread_cond_t x_cond;

/* Thread A (Imprime Hello) */
void *A (void *t) {
  printf("HELLO\n");

  pthread_mutex_lock(&x_mutex);
  x++; //incrementa contador de hellos
  if(x==2) { //se for o segundo hello, acorda quem espera
      pthread_cond_signal(&x_cond);
  }
  pthread_mutex_unlock(&x_mutex);

  pthread_exit(NULL);
}

/* Thread B (Espera 2 Hellos para imprimir Bye) */
void *B (void *t) {
  pthread_mutex_lock(&x_mutex);
  while (x < 2) {
     pthread_cond_wait(&x_cond, &x_mutex);
  }
  pthread_mutex_unlock(&x_mutex);

  printf("BYEBYE\n");

  pthread_exit(NULL);
}

int main(int argc, char *argv[]) {
  pthread_t threads[3];

  pthread_mutex_init(&x_mutex, NULL);
  pthread_cond_init(&x_cond, NULL);

  /* Criar 2 threads A e 1 thread B */
  pthread_create(&threads[0], NULL, A, NULL);
  pthread_create(&threads[1], NULL, A, NULL);
  pthread_create(&threads[2], NULL, B, NULL);

  for (int i = 0; i < 3; i++) {
    pthread_join(threads[i], NULL);
  }

  pthread_mutex_destroy(&x_mutex);
  pthread_cond_destroy(&x_cond);
  return 0;
}

Overwriting hello2bye.c


### **2. Execução**

Verifique se o "BYEBYE" só aparece após dois "HELLOs".

In [ ]:
!gcc -o hello2bye hello2bye.c -lpthread
!./hello2bye

HELLO
HELLO
BYEBYE


## **Atividade 4: Many to Many**

**Objetivo:** Duas threads A e duas threads B. As duas B devem esperar as duas A executarem "HELLO".

### **1. Implementação (Exercício)**

Complete o código abaixo para atender ao requisito.
**Dica:** O contador `x` deve chegar a 2, mas agora temos duas threads esperando. O que fazer?

### **2. Execução**

In [ ]:
!gcc -o atividade4 atividade4.c -lpthread
!./atividade4

cc1: fatal error: atividade4.c: No such file or directory
compilation terminated.
/bin/bash: line 1: ./atividade4: No such file or directory


## **Atividade 5: Aplicação em Soma (Entrega)**

**Objetivo:** Alterar o programa `soma-lock-atom.c` (do Lab 4). A thread `ExecutaTarefa` soma valores. A cada múltiplo de 1000, ela deve pausar e esperar uma thread `Extra` imprimir o valor atual.

### **Requisito do Código:**

1. Thread `ExecutaTarefa`: Soma números. Se `soma % 1000 == 0`, sinaliza a thread Extra e dorme.
2. Thread `Extra`: Dorme até receber sinal. Imprime a soma. Sinaliza `ExecutaTarefa` para continuar.

### **Espaço para Implementação**

Utilize o bloco abaixo para desenvolver a solução que será enviada.

In [ ]:
%%writefile lab5_entrega.c
/* Disciplina: Programacao Concorrente */
/* Prof.: Silvana Rossetto */
/* Codigo: Comunicação entre threads usando variável compartilhada e exclusao mutua com bloqueio */

#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>

long int soma = 0; //variavel compartilhada entre as threads
pthread_mutex_t mutex; //variavel de lock para exclusao mutua
pthread_cond_t print, count;

//funcao executada pelas threads
void *ExecutaTarefa (void *arg) {
  int id = *(int *) arg;
  printf("Thread : %d esta executando...\n", id);

  for (int i=0; i<100000; i++) {
     //--entrada na SC
     pthread_mutex_lock(&mutex);
     //--SC (seção critica)
     soma++; //incrementa a variavel compartilhada
     //--saida da SC
     if (!(soma%1000)) {
        printf("soma = %ld \n", soma);
        pthread_cond_signal(&print);
        pthread_cond_wait(&count, &mutex);
     }

     pthread_mutex_unlock(&mutex);
  }
  printf("Thread : %d terminou!\n", id);
  pthread_exit(NULL);
}

//funcao executada pela thread de log
void *extra (void *args) {
  printf("Extra : esta executando...\n");
  while (1) {
    pthread_cond_wait(&print, &mutex);
    printf("soma = %ld \n", soma);
    pthread_cond_signal(&count);
  }

  printf("Extra : terminou!\n");
  pthread_exit(NULL);
}

//fluxo principal
int main(int argc, char *argv[]) {
   pthread_t *tid; //identificadores das threads no sistema
   int nthreads; //qtde de threads (passada linha de comando)

   //--le e avalia os parametros de entrada
   if(argc<2) {
      printf("Digite: %s <numero de threads>\n", argv[0]);
      return 1;
   }
   nthreads = atoi(argv[1]);
   int id[nthreads];

   //--aloca as estruturas
   tid = (pthread_t*) malloc(sizeof(pthread_t)*(nthreads+1));
   if(tid==NULL) {puts("ERRO--malloc"); return 2;}

   //--inicilaiza o mutex (lock de exclusao mutua)
   pthread_mutex_init(&mutex, NULL);

   //--cria as threads
   for(int t=0; t<nthreads; t++) {
     id[t] = t;
     if (pthread_create(&tid[t], NULL, ExecutaTarefa, &id[t])) {
       printf("--ERRO: pthread_create()\n"); exit(-1);
     }
   }

   //--cria thread de log
   if (pthread_create(&tid[nthreads], NULL, extra, NULL)) {
      printf("--ERRO: pthread_create()\n"); exit(-1);
   }

   //--espera todas as threads terminarem
   for (int t=0; t<nthreads+1; t++) {
     if (pthread_join(tid[t], NULL)) {
         printf("--ERRO: pthread_join() \n"); exit(-1);
     }
   }

   //--finaliza o mutex
   pthread_mutex_destroy(&mutex);

   printf("Valor de 'soma' = %ld\n", soma);

   return 0;
}

Overwriting lab5_entrega.c



### **Testar Solução**

In [ ]:
!gcc -o lab5_entrega lab5_entrega.c -lpthread
!./lab5_entrega 2


Thread : 1 esta executando...
soma = 1000 
Thread : 2 esta executando...
Thread : 3 esta executando...
soma = 2000 
soma = 3000 
Thread : 4 esta executando...
Thread : 0 esta executando...
Extra : esta executando...
soma = 4000 
soma = 5000 
soma = 5000 
soma = 6000 
soma = 6000 
soma = 7000 
soma = 7000 
soma = 8000 
soma = 8000 
soma = 9000 
soma = 9000 
soma = 10000 
soma = 10000 
soma = 11000 
soma = 11000 
soma = 12000 
soma = 12000 
soma = 13000 
soma = 13000 
soma = 14000 
soma = 14000 
soma = 15000 
soma = 15000 
soma = 16000 
soma = 16000 
soma = 17000 
soma = 17000 
soma = 18000 
soma = 18000 
soma = 19000 
soma = 19000 
soma = 20000 
soma = 20000 
soma = 21000 
soma = 21000 
soma = 22000 
soma = 22000 
soma = 23000 
soma = 23000 
soma = 24000 
soma = 24000 
soma = 25000 
soma = 25000 
soma = 26000 
soma = 26000 
soma = 27000 
soma = 27000 
soma = 28000 
soma = 28000 
soma = 29000 
soma = 29000 
soma = 30000 
soma = 30000 
soma = 31000 
soma = 31000 
soma = 32000 
soma = 3200

## **Entrega**

Disponibilize o código implementado na **Atividade 5** em um ambiente de acesso remoto (GitHub ou GitLab). Use o formulário de entrega para enviar o link.
